# 02 — Modeling: Baseline → Random Forest → XGBoost

Core question: **Which model + imbalance strategy gives the best PR-AUC?**

Imbalance strategies tested:
- class_weight='balanced' (LR + RF)
- SMOTE oversampling
- scale_pos_weight (XGBoost)
- Combined: SMOTE + XGBoost

Key metric: **PR-AUC** (not accuracy — see notebook 01 for why)

Run time: ~5–10 minutes for all models

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, classification_report
import sys; sys.path.insert(0, '..')
from src.features import load_raw, build_features, get_X_y
from src.model import FailureClassifier, find_cost_optimal_threshold

%matplotlib inline
sns.set_theme(style='darkgrid')

In [2]:
df = build_features(load_raw())
X, y = get_X_y(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {y_train.shape[0]:,} | Test: {y_test.shape[0]:,}")
print(f"Train failure rate: {y_train.mean()*100:.1f}% | Test: {y_test.mean()*100:.1f}%")

[features] Loaded 10,000 rows | Failure rate: 3.4%
Train: 8,000 | Test: 2,000
Train failure rate: 3.4% | Test: 3.4%


In [3]:
# Baseline: Logistic Regression with class_weight='balanced'
lr_clf = FailureClassifier(model_type='logistic', use_smote=False)
lr_clf.fit(X_train, y_train)
lr_results = lr_clf.evaluate(X_test, y_test)
print("Logistic Regression (baseline):")
print(lr_results)

[model] Fitted logistic | optimal_threshold=0.53 | train_cost=$4,142,000
Logistic Regression (baseline):
{'threshold_used': 0.5250000000000001, 'roc_auc': 0.9369, 'pr_auc': 0.4549, 'f1': 0.3105, 'precision': 0.1891, 'recall': 0.8676, 'business_cost': 956000, 'optimal_threshold': 0.575, 'optimal_cost': 872000, 'confusion_matrix': [[1679, 253], [9, 59]], 'n_fn': 9, 'n_fp': 253}


In [4]:
# Random Forest with SMOTE
rf_clf = FailureClassifier(model_type='rf', use_smote=True)
rf_clf.fit(X_train, y_train)
rf_results = rf_clf.evaluate(X_test, y_test)
print("Random Forest + SMOTE:")
print(rf_results)

[model] After SMOTE: [7729 7729]


[model] Fitted rf | optimal_threshold=0.43 | train_cost=$0
Random Forest + SMOTE:
{'threshold_used': 0.42500000000000016, 'roc_auc': 0.9831, 'pr_auc': 0.8198, 'f1': 0.6821, 'precision': 0.5619, 'recall': 0.8676, 'business_cost': 542000, 'optimal_threshold': 0.175, 'optimal_cost': 500000, 'confusion_matrix': [[1886, 46], [9, 59]], 'n_fn': 9, 'n_fp': 46}


In [5]:
# XGBoost (scale_pos_weight + SMOTE)
xgb_clf = FailureClassifier(model_type='xgb', use_smote=True)
xgb_clf.fit(X_train, y_train)
xgb_results = xgb_clf.evaluate(X_test, y_test)
print("XGBoost + SMOTE:")
print(xgb_results)

[model] After SMOTE: [7729 7729]


/opt/anaconda3/envs/industrial-failure/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [12:38:35] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[model] Fitted xgb | optimal_threshold=0.83 | train_cost=$446,000
XGBoost + SMOTE:
{'threshold_used': 0.8250000000000003, 'roc_auc': 0.9793, 'pr_auc': 0.841, 'f1': 0.601, 'precision': 0.4519, 'recall': 0.8971, 'business_cost': 498000, 'optimal_threshold': 0.775, 'optimal_cost': 466000, 'confusion_matrix': [[1858, 74], [7, 61]], 'n_fn': 7, 'n_fp': 74}


In [6]:
# Comparison table
results_df = pd.DataFrame([
    {'Model': 'LR (baseline)', **{k: v for k,v in lr_results.items() if k in ['roc_auc','pr_auc','f1','recall','precision','business_cost']}},
    {'Model': 'RF + SMOTE',   **{k: v for k,v in rf_results.items() if k in ['roc_auc','pr_auc','f1','recall','precision','business_cost']}},
    {'Model': 'XGBoost + SMOTE', **{k: v for k,v in xgb_results.items() if k in ['roc_auc','pr_auc','f1','recall','precision','business_cost']}},
])
print(results_df.to_string(index=False))
# NOTE: save this table — it goes in README.md

          Model  roc_auc  pr_auc     f1  precision  recall  business_cost
  LR (baseline)   0.9369  0.4549 0.3105     0.1891  0.8676         956000
     RF + SMOTE   0.9831  0.8198 0.6821     0.5619  0.8676         542000
XGBoost + SMOTE   0.9793  0.8410 0.6010     0.4519  0.8971         498000


In [7]:
# Save best model
xgb_clf.save('../models/')
print("Model saved. Proceed to notebook 03 for threshold tuning.")

[model] Saved to ../models
Model saved. Proceed to notebook 03 for threshold tuning.
